# Stratified Analysis

Stratified analysis creates **separate control charts for each factor level**. This is powerful when:

- Each stream (machine, lane, operator) has its own behavior
- You want to detect changes within individual streams
- Comparing streams directly would mask within-stream signals

## What You'll Learn

1. Create stratified IMR charts for multiple streams
2. Understand when to use stratified vs. combined analysis
3. Navigate between individual stream charts
4. Detect signals within each stratum

## Setup

In [1]:
import numpy as np
import pandas as pd
from processbehavior import ProcessBehavior

## Create Multi-Stream Data

Simulate a filling line with 4 lanes, each with:
- Different baseline performance
- Different variation levels
- Lane C has a special cause event at time 15

In [2]:
np.random.seed(42)

lanes = ['Lane_A', 'Lane_B', 'Lane_C', 'Lane_D']
n_times = 20

# Each lane has different characteristics
lane_config = {
    'Lane_A': {'mean': 100, 'std': 1.0},
    'Lane_B': {'mean': 101, 'std': 1.5},
    'Lane_C': {'mean': 99, 'std': 1.2},
    'Lane_D': {'mean': 100.5, 'std': 0.8}
}

data = []
for t in range(n_times):
    for lane in lanes:
        config = lane_config[lane]
        
        # Special cause: Lane C at time 15
        special = 6 if (lane == 'Lane_C' and t == 14) else 0
        
        value = config['mean'] + special + np.random.normal(0, config['std'])
        data.append({
            'batch': t + 1,
            'lane': lane,
            'fillweight': round(value, 2)
        })

df = pd.DataFrame(data)
print(f"Dataset: {len(df)} observations")
print(f"Structure: {len(lanes)} lanes x {n_times} batches")
df.head(8)

Dataset: 80 observations
Structure: 4 lanes x 20 batches


,batch,lane,fillweight
0,1,Lane_A,100.50
1,1,Lane_B,100.79
2,1,Lane_C,99.78
3,1,Lane_D,101.72
4,2,Lane_A,99.77
5,2,Lane_B,100.65
6,2,Lane_C,100.90
7,2,Lane_D,101.11


## Formulate the Study

In [3]:
pb = ProcessBehavior(df)

study = pb.formulate(
    response=pb.cols.fillweight,
    factors=[pb.cols.lane],
    time=pb.cols.batch
)

print(f"SDS: {study.sds} ({study.sds_name})")
print(f"Valid charts: {study.valid_charts}")
print(f"Recommended: {study.recommended_chart}")

SDS: 1 (Full Factorial with Complete Replication)
Valid charts: ['Xbar', 'S', 'R', 'Imr']
Recommended: Xbar


## Combined vs. Stratified Analysis

With factors and time, you have two options:

### Combined Analysis (Xbar-S)
- Compares factor levels against each other
- Uses pooled within-group variance
- Good for detecting **between-group differences**

### Stratified Analysis (IMR per level)
- Each factor level gets its own chart
- Uses within-level variance for each
- Good for detecting **within-group changes over time**

## Create Stratified IMR Charts

In [ ]:
# Execute with IMR stratified by lane - creates one chart per lane
result = study.execute(chart='Imr', by=['lane'])

print(f"Charts created: {result.all_charts}")
print(f"Strata: {result.charts['Imr']['strata']}")

## View Individual Stream Charts

In [ ]:
# Get chart data - all lanes in one DataFrame with 'rsg' column for the lane
chart_data = result.get_chart('Imr')
print("Stratified Chart Data (all lanes):")
print(f"Total observations: {len(chart_data)}")
print()

# Filter to just Lane A
lane_a_data = chart_data[chart_data['rsg'] == 'Lane_A']
print("Lane A Data:")
lane_a_data.head(10)

In [ ]:
# Statistics are stored per stratum
stats = result.get_statistics('Imr')

for lane in lanes:
    s = stats[lane]
    print(f"{lane}: CL={s['center']:.2f}, UCL={s['upl']:.2f}, LCL={s['lpl']:.2f}")

## Visualize All Lanes Together

Use faceting to see all lanes in one view:

In [ ]:
# Faceted plot shows all lanes
fig = result.plot(
    chart='Imr',
    show_zones=True,
    highlight_signals=True
)
fig.show()

## View Individual Lane

In [ ]:
# Focus on Lane C (has the special cause)
# The plot shows all lanes as facets - Lane C's signal is visible
fig = result.plot(
    chart='Imr',
    show_zones=True,
    highlight_signals=True,
    show_rules=True
)
fig.show()

## Detect Signals Across All Lanes

In [ ]:
# Detect signals on stratified IMR chart
signals = result.detect_signals(chart='Imr')

if signals.has_signals:
    print(f"Total signals: {signals.count}")
    print()
    print("Violations:")
    display(signals.violations)
else:
    print("No signals detected")

## Iterate Through Charts

Use the iterator for programmatic access:

In [ ]:
# Get statistics for all lanes
stats = result.get_statistics('Imr')
chart_data = result.get_chart('Imr')

for lane in lanes:
    lane_data = chart_data[chart_data['rsg'] == lane]
    lane_stats = stats[lane]
    print(f"{lane}: n={len(lane_data)}, center={lane_stats['center']:.2f}")

## Compare with Combined Analysis

Let's see what the Xbar-S analysis shows:

In [11]:
# Combined Xbar analysis
result_xbar = study.execute(chart='Xbar')

fig = result_xbar.plot(
    chart='Xbar',
    show_zones=True,
    highlight_signals=True,
    title='Combined Xbar Chart'
)
fig.show()

### Key Difference

Notice how the **combined Xbar chart** may show different signals than the **stratified IMR charts**:

- Xbar uses pooled variance across lanes
- Lane-specific variation differences are averaged out
- A signal in one lane might be masked

The **stratified approach** uses each lane's own variation, making it more sensitive to within-lane changes.

## When to Use Each Approach

### Use Combined (Xbar-S) When:
- Comparing lanes/machines/operators to each other
- Looking for systematic differences between groups
- Setting up initial process capability

### Use Stratified (IMR) When:
- Monitoring individual streams over time
- Each stream has different inherent variation
- Want maximum sensitivity to within-stream changes
- Historical data shows streams behave differently

## Summary

In this tutorial, you learned:

- Stratified IMR creates one chart per factor level
- Each stratum has its own control limits
- Use faceted plots to view all strata together
- Stratified analysis is more sensitive to within-stream changes
- Choose stratified vs. combined based on your question

## Next Steps

- [Signal Detection](signal-detection.ipynb) - All Western Electric rules
- [Chart Types](../user-guide/chart-types.md) - When to use each chart type
- [Plotting & Themes](../user-guide/plotting.md) - Advanced visualization options